# Структура обработанного датасета DRP-372 (только образцы 256x256x256)

## Папки

- data/npy/ - бинарные 3D-структуры пород (файлы *.npy)
- data/csv/ - целевые значения проницаемости (файлы *.csv)

## Формат .npy (входные данные)

- Размерность = (256, 256, 256)
- Тип данных: float32 (0 - поровое пространство, 1 - твёрдая фаза)
- Загрузка: `vol = np.load("data/npy/204_03_256.npy")`

## Формат .csv (целевые переменные)

Каждый CSV в data/csv/ содержит 6 строк (давления 1,2,5,10,20 МПа и LBM):

| pressure | permeability |
|----------|--------------|
| 1        |       |
| 2        |       |
| 5        |       |
| 10       |       |
| 20       |      |
| LBM      |     |

- pressure: давление в МПа (для LBM - строка 'LBM')
- permeability: проницаемость (при отсутствии данных - NaN)

Также все csv объединены в один файл с именем "dataset_combined.csv", расположенный в корне проекта

## PyTorch Dataset

Класс PermDataset возвращает пару (volume, target_vector):
- volume: тензор (1, 256, 256, 256)
- target: тензор (6,)

```python
dataset = PermDataset()
vol, perm = dataset[0]  # vol.shape=(1,256,256,256), perm.shape=(6,)

In [1]:
!pip install remotezip

In [2]:
!pip install "numpy<2.0" "scikit-learn<1.5" ripser persim
import numpy as np
print(f"Current NumPy version: {np.__version__}")
if np.__version__.startswith('2.'):
    print("\n[!] ВНИМАНИЕ: Нужно перезапустить сессию (Runtime -> Restart session), чтобы изменения вступили в силу.")

np.set_printoptions(precision=2, suppress=True, linewidth=120)

Current NumPy version: 1.26.4


## Получение и объединение .csv файлов с результатами симуляций, преобразование .mat -> .npy

In [3]:

# @title
from remotezip import RemoteZip
import h5py
import pandas as pd
import os
import torch
from torch.utils.data import Dataset
import fnmatch
from tqdm.notebook import tqdm




url = "https://web.corral.tacc.utexas.edu/digitalporousmedia/archive/DRP-372/DRP-372_archive.zip"
os.makedirs("data/npy", exist_ok=True)
os.makedirs("data/csv", exist_ok=True)

def cb(name, obj):
  if isinstance(obj, h5py.Dataset) and obj.ndim == 3:
      return obj

def find_3d_dataset(h5_file):
  """Возвращает 3D массив или None."""
  ds = h5_file.visititems(cb)
  return ds

def get_permeability(csv_bytes):
  """Извлекает значение permeability из CSV."""
  try:
      text = csv_bytes.decode('utf-8')
      for line in text.splitlines():
          if 'permeability' in line.lower():
              parts = line.split(',')
              if len(parts) >= 2:
                  val = parts[1].strip()
                  if not val and len(parts) > 0:
                      val = parts[0].strip()
                  return float(val)
      return None
  except Exception as e:
      print(f"Ошибка парсинга: {e}")
      return None

with RemoteZip(url) as rz:
    all_files = rz.namelist()
    mat_files = [f for f in all_files if '_256.mat' in f and f.endswith('.mat')]

    for mat_path in tqdm(mat_files):
        name = mat_path.split('/')[-1][:-4]
        base_dir = '/'.join(mat_path.split('/')[:-1])

        #.mat -> .npy
        npy_path = f"data/npy/{name}.npy"
        if not os.path.exists(npy_path):
            try:
                rz.extract(mat_path, path='.')
                with h5py.File(mat_path, 'r') as hf:
                    vol = find_3d_dataset(hf)
                    if vol is not None:
                        np.save(npy_path, vol)
                os.remove(mat_path)
            except Exception as e:
                print(f"Ошибка при обработке {name}: {e}")
                continue

        # Поиск permeability
        perms = []
        pressures = [1,2,5,10,20,'LBM']
        for p in pressures:
            if p == 'LBM':
                pattern = f"{base_dir}/Single Phase*/LBM.csv"
            else:
                pattern = f"{base_dir}/Single Phase*/P_{p}_MPa.csv"
            matching = fnmatch.filter(all_files, pattern)
            if matching:
                csv_path = matching[0]
                try:
                    content = rz.read(csv_path)
                    perm = get_permeability(content)
                    perms.append(perm)
                except Exception as e:
                    print(f"Не удалось прочитать {csv_path}: {e}")
                    perms.append(None)
            else:
                perms.append(None)

        df_out = pd.DataFrame({'pressure': pressures, 'permeability': perms})
        df_out.to_csv(f"data/csv/{name}.csv", index=False)
        print(f"{name}: {perms}")


  0%|          | 0/136 [00:00<?, ?it/s]

317_02_256: [1.7531069478361092, 0.7624153232108687, 0.3672116813928819, 0.22045933940895926, 0.12627494639751016, 1.2627494639751017e-13]
374_02_03_256: [41.71824200766815, 23.39382825179733, 14.81054390724986, 10.79750869681798, 7.3604824660096755, 7.360482466009675e-12]
374_03_015_256: [11.077355113132537, 5.456470794585234, 2.813152775616743, 1.7521527300326838, 1.0502429365884796, 1.0502429365884795e-12]
374_09_02_256: [0.990834308981556, 0.44075643750433957, 0.19631521681851785, 0.10782334746938196, 0.05669857380200553, 5.669857380200552e-14]
374_07_02_256: [4.046186118928552, 1.8986294914880073, 0.9141169494090212, 0.5370872534526243, 0.30332613147969517, 3.0332613147969516e-13]
204_03_256: [0.2684027965832713, 0.12569862779265922, 0.04668785990902349, 0.02139093769971679, None, None]
65_03_256: [2.1348239560183573, 1.0288814472837888, 0.4769045668374405, 0.26819575440528665, 0.14353657116169724, 1.4353657116169723e-13]
10_01_256: [0.24533081259590386, 0.10793278113277016, 0.050

## Объединение .csv файлов в один и сохранение в корень

In [4]:
# @title
csv_dir = "data/csv"
output_file = "dataset_combined.csv"

pressures = ['1','2','5','10','20','LBM']

rows = []
for idx, fname in enumerate(sorted(os.listdir(csv_dir)), start=1):
    if not fname.endswith(".csv"):
        continue
    sample_name = fname[:-4]
    df = pd.read_csv(os.path.join(csv_dir, fname))
    df['pressure'] = df['pressure'].astype(str)
    perm_dict = dict(zip(df['pressure'], df['permeability']))
    perms = [perm_dict.get(p, None) for p in pressures]
    perms = [float(x) if x is not None else None for x in perms]
    rows.append([idx, sample_name] + perms)

columns = ["index", "sample"] + [f"perm_{p}MPa" if p != 'LBM' else "perm_LBM" for p in pressures]
result_df = pd.DataFrame(rows, columns=columns)
result_df.to_csv(output_file, index=False)
print(f"Сохранён {output_file} с {len(result_df)} строками")


Сохранён dataset_combined.csv с 133 строками


## Создание класса датасета, унаследованного от torch.utils.data.Dataset

In [5]:
# @title
class PermeabilityDataset(Dataset):
    def __init__(self, npy_dir='data/npy', csv_dir='data/csv'):
        self.npy_dir = npy_dir
        self.csv_dir = csv_dir
        samples = [f[:-4] for f in os.listdir(npy_dir) if f.endswith('.npy')]
        samples = [s for s in samples if os.path.exists(f"{csv_dir}/{s}.csv")]

        # Функция для извлечения чисел из названия для правильной числовой сортировки
        def numerical_sort_key(name):
            parts = name.split('_')
            return [int(parts[0]), int(parts[1]), int(parts[2])]


        self.samples = sorted(samples, key=numerical_sort_key)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        vol = np.load(f"{self.npy_dir}/{name}.npy")
        vol_tensor = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)
        df = pd.read_csv(f"{self.csv_dir}/{name}.csv")
        perm = df['permeability'].values.astype(np.float32)
        perm = np.nan_to_num(perm, nan=0.0)
        return vol_tensor, torch.tensor(perm, dtype=torch.float32), name

In [6]:
# @title
dataset = PermeabilityDataset()
print(f'перввый куб {dataset[0][2]}')
print(f'последний куб {dataset[132][2]}')
print(f"Размер датасета: {len(dataset)}")
if len(dataset) > 0:
    sample_vol, sample_target, sample_name = dataset[0]
    print(f"Пример volume: {sample_vol}")
    print(f"Пример target: {sample_target}")
    print(f"Пример имени образца: {sample_name}")




перввый куб 10_01_256
последний куб 374_10_03_256
Размер датасета: 133
Пример volume: tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         ...,

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
        

# Скачиваем библиотеку PyVista
Данная библиотека предоставляет инструменты для отрисовки 3-D моделей

In [7]:
pip install pyvista numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 6.0 MB/s eta 0:00:00


# Фильтрация бинарных рисунков
Фнкции фильтрации позволяют покрасить пустоты в
 градиент, что в дальнейшем позволит делать градацию серого и считать персистентные гомологии.

In [8]:
# @title Подключаем библиотеки
import pyvista as pv
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import matplotlib.image as mpimg

## Функция визуализации породы
Данная функция позволяет отрисовать изображение породы по трехмерному бинарному массиву данных

In [9]:
# @title Функция визуализации
def visualize(data, file_name='rock_structure_1',
              L = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, decimation_factor=0.9, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape

    # Создаем сетку
    grid = pv.ImageData(dimensions=shape, spacing=(1, 1, 1), origin=(0, 0, 0))
    grid.point_data["values"] = data_np.flatten(order="F")

    # Генерация изоповерхности (максимальное количество полигонов на входе)
    surface_mesh = grid.contour([0.5], scalars="values")

    # Агрессивное уменьшение количества полигонов (Decimation)
    if decimation_factor > 0 and surface_mesh.n_cells > 0:
        surface_mesh = surface_mesh.decimate(decimation_factor)

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Рисуем границы куба
    plotter.add_mesh(surface_mesh, color='grey', lighting=True) ## smooth_shading=True добавлять?
    plotter.add_mesh(pv.Cube(bounds=[0, shape[0]-1, 0, shape[1]-1, 0, shape[2]-1]),
                     style='wireframe', color='black', line_width=1, opacity=0.5) # добавляем границы

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L
    plotter.add_title(file_name, font_size=10)  # название камня

    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [10]:
# @title убираемм картинки временные
import os
import glob

def clear_tmp_images(extension="png"):
    """Удаляет временные изображения из /tmp/"""
    files = glob.glob(f"/tmp/*.{extension}")
    for f in files:
        try:
            os.remove(f)
        except OSError as e:
            print(f"Ошибка при удалении {f}: {e}")
    print(f"Очищено файлов: {len(files)}")

# Пример использования:
clear_tmp_images()

Очищено файлов: 0


In [11]:
# @title ДЛЯ ОЗУ
'''import psutil
import os
import sys
import pandas as pd

# 1. Общая статистика системы
!free -h

# 2. Поиск самых больших переменных в памяти Python
def get_size(obj, seen=None):
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum([get_size(v, seen) for v in obj.values()])
        size += sum([get_size(k, seen) for k in obj.keys()])
    elif hasattr(obj, '__dict__'):
        size += get_size(obj.__dict__, seen)
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
        size += sum([get_size(i, seen) for i in obj])
    return size

# Вывод топ-10 переменных
local_vars = globals().copy()
mem_usage = []
for var_name, var_val in local_vars.items():
    if not var_name.startswith('_'):
        try:
            size = sys.getsizeof(var_val) / (1024**2) # MB
            mem_usage.append({'Variable': var_name, 'Size (MB)': size})
        except:
            pass

df_mem = pd.DataFrame(mem_usage).sort_values('Size (MB)', ascending=False).head(10)
print("\nТоп-10 переменных в памяти:")
display(df_mem)'''

'import psutil\nimport os\nimport sys\nimport pandas as pd\n\n# 1. Общая статистика системы\n!free -h\n\n# 2. Поиск самых больших переменных в памяти Python\ndef get_size(obj, seen=None):\n    size = sys.getsizeof(obj)\n    if seen is None:\n        seen = set()\n    obj_id = id(obj)\n    if obj_id in seen:\n        return 0\n    seen.add(obj_id)\n    if isinstance(obj, dict):\n        size += sum([get_size(v, seen) for v in obj.values()])\n        size += sum([get_size(k, seen) for k in obj.keys()])\n    elif hasattr(obj, \'__dict__\'):\n        size += get_size(obj.__dict__, seen)\n    elif hasattr(obj, \'__iter__\') and not isinstance(obj, (str, bytes, bytearray)):\n        size += sum([get_size(i, seen) for i in obj])\n    return size\n\n# Вывод топ-10 переменных\nlocal_vars = globals().copy()\nmem_usage = []\nfor var_name, var_val in local_vars.items():\n    if not var_name.startswith(\'_\'):\n        try:\n            size = sys.getsizeof(var_val) / (1024**2) # MB\n            me

In [12]:
# @title ДЛЯ ОЗУ
'''# Проверим реальный размер в памяти для разных типов
shape = (256, 256, 256)
array_f64 = np.zeros(shape, dtype=np.float64)
array_f32 = np.zeros(shape, dtype=np.float32)

print(f"Размер float64 (default): {array_f64.nbytes / 1024**2:.1f} MB")
print(f"Размер float32: {array_f32.nbytes / 1024**2:.1f} MB")
print(f"Текущий тип массива height_filt: {height_filt.dtype}")'''

'# Проверим реальный размер в памяти для разных типов\nshape = (256, 256, 256)\narray_f64 = np.zeros(shape, dtype=np.float64)\narray_f32 = np.zeros(shape, dtype=np.float32)\n\nprint(f"Размер float64 (default): {array_f64.nbytes / 1024**2:.1f} MB")\nprint(f"Размер float32: {array_f32.nbytes / 1024**2:.1f} MB")\nprint(f"Текущий тип массива height_filt: {height_filt.dtype}")'

In [13]:
# @title Функции визуализации фильтраций 3D
def visualize_color_rock(data, filtration_vol, file_name='rock_structure_colored',
                         L = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, decimation_factor=0.9, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape
    filt_np = np.flip(filtration_vol.squeeze(), axis=0)

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Записываем маску породы и значения фильтрации в данные ячеек (cell_data)
    grid.cell_data["rock_mask"] = data_np.flatten(order="F").astype(np.uint8)
    grid.cell_data["dist_value"] = filt_np.flatten(order="F")

    # Переносим данные из ячеек в точки (point_data), так как алгоритм contour работает с точками
    grid = grid.cell_data_to_point_data()

    # Создаем полигональную сетку (изоповерхность) на границе пора/порода (значение 0.5)
    mesh = grid.contour([0.5], scalars="rock_mask")

    # Уменьшаем количество полигонов для ускорения отрисовки, если задан коэффициент
    if decimation_factor > 0 and mesh.n_cells > 0:
        mesh = mesh.decimate(decimation_factor)

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Добавляем сетку в визуализатор, раскрашивая по значениям фильтрации, убираем шкалу
    if "dist_value" in mesh.point_data:
        plotter.add_mesh(mesh, scalars="dist_value", cmap="jet", lighting=True, show_scalar_bar=False)
    else:
        # Если данные потерялись при децимации, интерполируем их заново
        mesh = mesh.sample(grid)
        plotter.add_mesh(mesh, scalars="dist_value", cmap="jet", lighting=True, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(pv.Cube(bounds=[0, shape[0]-1, 0, shape[1]-1, 0, shape[2]-1]),
                     style='wireframe', color='black', opacity=0.5)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))


def visualize_color_pore(data, filtration_vol, file_name='pore_cube_faces',
                         L=[(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape
    filt_np = np.flip(filtration_vol.squeeze(), axis=0)

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Если порода (data == 0), присваиваем -0.1 для серого цвета
    vis_values = np.where(data_np == 1, filt_np, -0.1)
    grid.cell_data["colors"] = vis_values.flatten(order="F")

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Извлекаем только внешние грани куба
    outline = grid.outline()
    faces = grid.extract_surface(algorithm='dataset_surface')

    # Создаем модифицированный jet
    jet_colors = plt.get_cmap('jet')(np.linspace(0, 1, 256))
    new_colors = np.vstack([np.array([106/255, 104/255, 99/255, 1.0]), jet_colors])
    custom_cmap = LinearSegmentedColormap.from_list('jet_with_grey', new_colors)

    plotter.add_mesh(faces, scalars="colors", cmap=custom_cmap, clim=[-0.1, 1.0],
                     lighting=False, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(outline, color="black", line_width=1)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [14]:
# @title Функции отрисовки срезов
def show_slice(data, k=128, axis = 'z', T = True):
    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = data.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = data.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = data.squeeze()[:, :, k]

    if not T:
      return slice_2d

    # Визуализация среза
    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=ListedColormap(['#6a6863', 'white'])) # 1-белый (поры), 0-серый (порода)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def show_slice_filtered_pore(data, normalized_grad, k=128, axis = 'z', T = True):
    # Применяем маску: там где порода (data == 0), ставим -0.1 для серого цвета
    vis_volume = np.where(data.squeeze() == 1, normalized_grad.squeeze(), -0.1)

    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = vis_volume[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = vis_volume[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = vis_volume[:, :, k]

    if not T:
      return slice_2d

    # Создаем копию jet, но добавляем цвет #6a6863 для породы
    base_cmap = plt.get_cmap('jet')
    new_colors = base_cmap(np.linspace(0, 1, 256))
    custom_cmap = LinearSegmentedColormap.from_list('custom_jet', new_colors)
    custom_cmap.set_under('#6a6863')

    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=custom_cmap, vmin=0, vmax=1)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def show_slice_filtered_rock(data, normalized_grad, k=128, axis = 'z', T = True):
    # Применяем маску: там где поры (data == 1), ставим -0.1 для белого цвета
    vis_volume = np.where(data.squeeze() == 0, normalized_grad.squeeze(), -0.1)

    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = vis_volume[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = vis_volume[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = vis_volume[:, :, k]

    if not T:
      return slice_2d

    # Создаем копию jet, но для пор устанавливаем белый цвет
    base_cmap = plt.get_cmap('jet')
    new_colors = base_cmap(np.linspace(0, 1, 256))
    custom_cmap = LinearSegmentedColormap.from_list('custom_jet_rock', new_colors)
    custom_cmap.set_under('white')

    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=custom_cmap, vmin=0, vmax=1)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()

In [ ]:
# @title Общие параметры
n = 1
vol_tensor, target, sample_name = dataset[n]
data = vol_tensor.squeeze().numpy().astype(np.uint8)

print(f'Имя файла {sample_name}, размер {data.shape}') ## выяснить из статьи нормальные названия и вытащить их
print(f'Пористость породы {int(np.sum(data)) / 256**3}') ## почему 1 у нас - это поры ???
visualize(data, sample_name)


Имя файла 16_01_256, размер (256, 256, 256)
Пористость породы 0.6415016055107117


# Функции фильтраций

In [ ]:
# @title Height filtration
def apply_height_filtration(shape, normal=(0, 0, 1), d=0):
    # Создаем сетку координат, центрированную относительно центра куба
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]
    z_c, y_c, x_c = z - (shape[0] // 2 - 1), y - (shape[0] // 2 - 1), x - (shape[0] // 2 - 1)

    # Нормируем вектор нормали
    n = np.array(normal) / np.linalg.norm(normal)

    # Вычисляем расстояние от каждой точки до плоскости
    dist_from_plane = x_c * n[0] + y_c * n[1] + z_c * n[2] - d
    filtration_values = np.abs(dist_from_plane)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad


In [ ]:
# @title Line filtration (New)
def apply_line_filtration(shape, p1=(0, 0, 0), p2=(255, 255, 255)):
    # Создаем сетку координат
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]
    p1 = np.array(p1)
    p2 = np.array(p2)

    # Векторы от p1 до всех точек сетки
    dx, dy, dz = x - p1[0], y - p1[1], z - p1[2]

    # Направляющий вектор прямой (p2 - p1) и его норма
    v = p2 - p1
    v_norm_sq = np.sum(v**2) + 1e-9

    # Расстояние от точки до прямой
    cross_x = dy * v[2] - dz * v[1]
    cross_y = dz * v[0] - dx * v[2]
    cross_z = dx * v[1] - dy * v[0]

    filtration_values = np.sqrt((cross_x**2 + cross_y**2 + cross_z**2) / v_norm_sq)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad

In [ ]:
# @title Radial filtration
def apply_radial_filtration(shape, p=(127, 127, 127)):
    # Создаем сетку координат
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]

    # Вычисляем квадраты расстояний от центра p для каждой оси
    dist_sq = (x - p[0])**2 + (y - p[1])**2 + (z - p[2])**2
    filtration_values = np.sqrt(dist_sq)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad

In [ ]:
# @title Функция визуализации фильтрации
def show_filtration(data, filt, sample_name='rock_filtration_1', axis = 'z', levels = [0, 127, 255],
                    camera_position = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)]):
    # Предварительная генерация 3D скриншотов
    data=data.squeeze()
    filt=filt.squeeze()
    visualize(data, sample_name, camera_position, False)
    visualize_color_rock(data, filt, f'rock_{sample_name}', camera_position, False)
    visualize_color_pore(data, filt, f'pore_{sample_name}', camera_position, False)

    # Создаем общую фигуру 4x3
    fig = plt.figure(figsize=(18, 24))

    # заголовок
    fig.text(0.15, 0.91, f"Образец: {sample_name}", fontsize=20, va='center')

    # шкала min max
    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=1))
    cax = fig.add_axes([0.42, 0.90, 0.53, 0.015])
    cb = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cb.set_label('Filtration values', labelpad=-75, fontsize=18)
    cb.set_ticks([0, 1])
    cb.set_ticklabels(['min', 'max'], fontsize=18)

    # 3D визуализация
    paths = [f'/tmp/{sample_name}.png', f'/tmp/rock_{sample_name}.png', f'/tmp/pore_{sample_name}.png']
    for i, path in enumerate(paths):
        ax = fig.add_subplot(4, 3, i + 1)
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')


    for i, k in enumerate(levels):
        row_idx = i + 1

        # Колонка 1: show_slice
        ax_bin = fig.add_subplot(4, 3, row_idx * 3 + 1)
        ax_bin.imshow(data[:, k, :] if axis == 'z' else (data[k, :, :] if axis == 'x' else data[:, :, k]), cmap=ListedColormap(['#6a6863', 'white']))
        ax_bin.axis('off')
        # Уменьшен отступ (x=-0.05 вместо -0.15)
        ax_bin.text(-0.05, 0.5, f"{axis} = {k}", transform=ax_bin.transAxes,
                    rotation='vertical', va='center', ha='right', fontsize=16)

        # Колонка 2: show_slice_filtered_rock
        ax_rock = fig.add_subplot(4, 3, row_idx * 3 + 2)
        vis_rock = np.where(data == 0, filt, -0.1)
        rock_cmap = LinearSegmentedColormap.from_list('rock_jet', plt.get_cmap('jet')(np.linspace(0, 1, 256)))
        rock_cmap.set_under('white')
        ax_rock.imshow(vis_rock[:, k, :] if axis == 'z' else (vis_rock[k, :, :] if axis == 'x' else vis_rock[:, :, k]), cmap=rock_cmap, vmin=0, vmax=1)
        ax_rock.axis('off')

        # Колонка 3: show_slice_filtered_pore
        ax_pore = fig.add_subplot(4, 3, row_idx * 3 + 3)
        vis_pore = np.where(data == 1, filt, -0.1)
        pore_cmap = LinearSegmentedColormap.from_list('pore_jet', plt.get_cmap('jet')(np.linspace(0, 1, 256)))
        pore_cmap.set_under('#6a6863')
        ax_pore.imshow(vis_pore[:, k, :] if axis == 'z' else (vis_pore[k, :, :] if axis == 'x' else vis_pore[:, :, k]), cmap=pore_cmap, vmin=0, vmax=1)
        ax_pore.axis('off')

    # Настраиваем отступы вручную, чтобы убрать пустоту справа
    plt.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.05, wspace=0.05, hspace=0.1)
    plt.show()

In [ ]:
# @title ДЛЯ ОЗУ

'''height_filt = height_filt.astype(np.float32)
line_filt = line_filt.astype(np.float32)
radial_filt = radial_filt.astype(np.float32)

# Если у вас есть исходные данные (бинарная маска 0 и 1),
# их можно перевести в uint8 (всего 1 байт на число, 16 МБ на весь куб 256^3)
data = data.astype(np.uint8)

print(f"Новый тип height_filt: {height_filt.dtype}")
print(f"Новый размер height_filt: {height_filt.nbytes / 1024**2:.1f} MB")'''

In [ ]:
# @title Запуск отображения фильтраций
height_filt = apply_height_filtration((256, 256, 256), normal=(1, 1, 1), d=100).astype(np.float32)
line_filt = apply_line_filtration((256, 256, 256), (127, 127, 127), (255, 255, 255)).astype(np.float32)
radial_filt = apply_radial_filtration((256, 256, 256), (127, 127, 127)).astype(np.float32)

#visualize(data, sample_name)
#visualize_color_rock(data, height_filt, f'rock_{sample_name}')
#visualize_color_pore(data, height_filt, f'pore_{sample_name}')

#show_slice(data, 0)
#show_slice_filtered_rock(height_filt, 0)
#show_slice_filtered_pore(height_filt, 0)


show_filtration(data, radial_filt, sample_name)

# Подсчёт персистентных диаграмм и визуализация кубического комплекса
В этом разделе мы считаем персистентные диаграммы при помощи методов библиотеки GUDHI для уже отфильтрованных в предыдущем разделе массивов, а также визуализуруем кибические комплексы разными способами.

In [ ]:
!pip install gudhi
!pip install giotto-tda
!pip install ripser

In [ ]:
# @title Подключаем библиотек 2
%matplotlib inline

import scipy as sp

from sklearn.datasets import make_circles

# gudhi
import gudhi as gd
from gudhi import bottleneck_distance
from gudhi.hera import wasserstein_distance
from gudhi.representations import BettiCurve, PersistenceImage, Landscape

# giotto
import gtda
from gtda.homology import VietorisRipsPersistence
# from gtda.diagrams import BettiCurve, PersistenceImage, PersistenceLandscape
# Для работы с кубикальными комплексами (изображениями)
from gtda.homology import CubicalPersistence
# Для визуализации диаграмм
from gtda.plotting import plot_diagram
# Для подготовки данных (бинаризация, фильтрация)
from gtda.images import Binarizer, HeightFiltration
import gtda.plotting as gtdaplot

# ripser
from ripser import lower_star_img
from ripser import Rips


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

import torch
import torch.nn as nn
import torch.nn.functional as F
import time

from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.nn import CrossEntropyLoss


from persim import plot_diagrams


In [ ]:
# @title Функции визуализации grayscale
def show_slice_grayscale(grad_data, k=40, axis = 'z', T = True):
    if axis == 'z':
      # Извлекаем срез по оси z
      grayscale_2d = grad_data.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      grayscale_2d = grad_data.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      grayscale_2d = grad_data.squeeze()[:, :, k]

    if not T:
      return grayscale_2d

    plt.figure(figsize=(8, 8))
    plt.imshow(grayscale_2d, cmap='gray') # gray 0 -> черный, 1 -> белый
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def visualize_grayscale_faces(grad_data, file_name='grayscale_pore_cube_faces',
                         camera_position=[(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, window_size=[635, 635]):
    grad_data_np = np.flip(grad_data.squeeze(), axis=0)
    shape = grad_data_np.shape

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Добавляем данные в сетку
    grid.cell_data["colors"] = grad_data_np.flatten(order="F")

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Извлекаем только внешние грани куба
    outline = grid.outline()
    faces = grid.extract_surface(algorithm='dataset_surface')

    # Теперь массив 'colors' существует в данных поверхности
    plotter.add_mesh(faces, scalars="colors", cmap='gray',
                     lighting=False, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(outline, color="black", line_width=1)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = camera_position

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [ ]:
# @title Кубик 85 на 85 на 85
grayscale_filt = radial_filt[np.newaxis, :85, :85, :85] # значения от 0 до 1 #:85, 85:170, 171:256
grayscale_data = vol_tensor.numpy()[:, :85, :85, :85] # 0 - порода, 1 - поры
small_sample_name = f'{sample_name}_111_radial'


# Передаем уровни (levels), которые не превышают размер 85
print(grayscale_filt.shape)
print(grayscale_data.shape)
show_filtration(grayscale_data, grayscale_filt, small_sample_name, levels=[0, 40, 84], camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)])

In [ ]:
# @title Разные раскраски серого
grayscale_only_pore = np.where(grayscale_data == 1, grayscale_filt, 1) # порода белая
grayscale_only_rock = np.where(grayscale_data == 0, grayscale_filt, 1) # поры белые
grayscale_all = np.where(grayscale_data == 1, grayscale_filt / 2, (grayscale_filt + 1) / 2) # порода 0 - 1/2, поры 1/2 - 1, то есть порода темнее

print(grayscale_only_pore.shape)
print(grayscale_only_rock.shape)
print(grayscale_all.shape)

fig, ax = plt.subplots(ncols=3, nrows=2, figsize=(10, 10), dpi=200)


visualize_grayscale_faces(grayscale_only_pore, f'pore_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
visualize_grayscale_faces(grayscale_only_rock, f'rock_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
visualize_grayscale_faces(grayscale_all, f'multipl_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)

ax[0, 0].imshow(mpimg.imread(f'/tmp/pore_{small_sample_name}.png'))
ax[0, 0].set_title("Только поры")
ax[0, 0].axis('off')

ax[0, 1].imshow(mpimg.imread(f'/tmp/rock_{small_sample_name}.png'))
ax[0, 1].set_title("Только порода")
ax[0, 1].axis('off')

ax[0, 2].imshow(mpimg.imread(f'/tmp/multipl_{small_sample_name}.png'))
ax[0, 2].set_title("Всё")
ax[0, 2].axis('off')

ax[1, 0].imshow(show_slice_grayscale(grayscale_only_pore, 84, T = False), cmap='gray') # gray 0 -> черный, 1 -> белый
ax[1, 0].axis('off')
ax[1, 0].set_title("Срез z = 84")

ax[1, 1].imshow(show_slice_grayscale(grayscale_only_rock, 84, T = False), cmap='gray')
ax[1, 1].axis('off')
ax[1, 1].set_title("Срез z = 84")

ax[1, 2].imshow(show_slice_grayscale(grayscale_all, 84, T = False), cmap='gray')
ax[1, 2].axis('off')
ax[1, 2].set_title("Срез z = 84")

plt.tight_layout()
plt.subplots_adjust(wspace=0.05, hspace=-0.43)
plt.show()

In [ ]:

# HeightFiltration ожидает на вход набор изображений в формате (n_samples, h, w) или (n_samples, d, h, w)
# У нас один куб 'data' с размерностью (256, 256, 256)
# Добавим размерность для n_samples:
input_data = data[np.newaxis, :, :, :]

# Инициализируем фильтрацию по высоте.
# Параметр direction определяет вектор, вдоль которого будет расти значение фильтрации.
hf = HeightFiltration(direction=np.array([0, 0, 1]))

# Применяем трансформацию
filtered_images = hf.fit_transform(input_data)

print(f"Форма входных данных: {input_data.shape}")
print(f"Форма выходных данных: {filtered_images.shape}")

# Теперь filtered_images[0] содержит значения фильтрации, которые можно подать в CubicalPersistence

In [ ]:
# @title Функции отрсовки персистентных диаграмм
def fast_show_dg_ax(dg, ax_obj, title):
    ax_obj.set_title(title)
    ax_obj.set_xlim(-0.05, 1.05)
    ax_obj.set_ylim(-0.05, 1.05)
    ax_obj.set_aspect('equal')


    ax_obj.scatter(dg[dg[:, 2] == 0, 0], dg[dg[:, 2] == 0, 1], c="#ef553b", alpha=0.6, label="H0", s=20)
    ax_obj.scatter(dg[dg[:, 2] == 1, 0], dg[dg[:, 2] == 1, 1], c="#00cc96", alpha=0.6, label="H1", s=20)
    ax_obj.scatter(dg[dg[:, 2] == 2, 0], dg[dg[:, 2] == 2, 1], c="#ab63fa", alpha=0.6, label="H2", s=20)

    ax_obj.plot([-0.05, 1.1], [-0.05, 1.1], c="k", linestyle="--", linewidth=1, alpha=0.3)
    ax_obj.plot([-0.05, 1.1], [1, 1], c="k", linestyle="--", linewidth=1, alpha=0.3)
    ax_obj.set_xlabel("Birth")
    ax_obj.set_ylabel("Death")
    ax_obj.legend(loc="lower right")


def fast_show_dg(dg):
    plt.figure(figsize=(5, 5))

    plt.title("Персистентная диаграмма")
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)

    plt.scatter(dg[dg[:, 2] == 0, 0], dg[dg[:, 2] == 0, 1], c="#ef553b", alpha=0.6, label="H0", s=20)
    plt.scatter(dg[dg[:, 2] == 1, 0], dg[dg[:, 2] == 1, 1], c="#00cc96", alpha=0.6, label="H1", s=20)
    plt.scatter(dg[dg[:, 2] == 2, 0], dg[dg[:, 2] == 2, 1], c="#ab63fa", alpha=0.6, label="H2", s=20)

    plt.plot([-0.05, 1.1], [-0.05, 1.1], c="k", linestyle="--", linewidth=1, alpha=0.3)
    plt.plot([-0.05, 1.1], [1, 1], c="k", linestyle="--", linewidth=1, alpha=0.3)
    plt.xlabel("Birth")
    plt.ylabel("Death")
    plt.legend(loc="lower right")

In [ ]:
# @title Подсчет персистентных диграмм giotto (дольше в 1.5 раза)
'''cp = CubicalPersistence(homology_dimensions=[0, 1, 2], n_jobs=-1)
# Вычисляем диаграмму персистентности
diagrams = cp.fit_transform(grayscale_all) # Shape: (n_samples, total top features, [Birth, Death, Dimension])
# Расчет для трех случаев (используем ранее созданные переменные)
dg_pore = cp.fit_transform(grayscale_only_pore)[0]
dg_rock = cp.fit_transform(grayscale_only_rock)[0]
dg_all = cp.fit_transform(grayscale_all)[0]

# красивый вывод

fig = plot_diagram(diagrams[0])
fig.show()

# быстрый вывод
#fast_show_dg(diagrams[0])



fig, ax = plt.subplots(ncols=3, nrows=1, figsize=(15, 5))

fast_show_dg_ax(dg_pore, ax[0], "Только поры")
fast_show_dg_ax(dg_rock, ax[1], "Только порода")
fast_show_dg_ax(dg_all, ax[2], "Всё")

plt.tight_layout()
plt.show()

'''

In [ ]:
# @title Подсчет персистентных диграмм GUDHI
def gudhi_to_array(diagram):
    a = np.array([[birth, death, dim] for (dim, (birth, death)) in diagram])
    a[np.isinf(a[:, 1]), 1] = 1.0
    return a


def compute_gudhi_dg(data_cube):
    cc = gd.CubicalComplex(dimensions=data_cube.shape, top_dimensional_cells=data_cube.flatten())
    return gudhi_to_array(cc.persistence())



# Расчет для трех случаев через GUDHI
dg_pore_gd = compute_gudhi_dg(grayscale_only_pore[0]) # Shape: (total top. features, [Birth, Death, Dimension])
dg_rock_gd = compute_gudhi_dg(grayscale_only_rock[0])
dg_all_gd = compute_gudhi_dg(grayscale_all[0])



# красивый вывод при помощи giotto
fig = plot_diagram(dg_pore_gd)
fig.show()

# быстрый вывод
#fast_show_dg(dg_pore_gd)

# Визуализация результатов GUDHI
fig, ax = plt.subplots(ncols=3, nrows=1, figsize=(15, 5), dpi=150)
fast_show_dg_ax(dg_pore_gd, ax[0], "Только поры")
fast_show_dg_ax(dg_rock_gd, ax[1], "Только порода")
fast_show_dg_ax(dg_all_gd, ax[2], "Всё")

plt.tight_layout()
plt.show()

In [ ]:
# @title Функции визуализации кубического комплекса
def show_cubical_complex_slice(img_filtered, k=42, axis = 'z', num_thresholds = 10):
    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = img_filtered.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = img_filtered.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = img_filtered.squeeze()[:, :, k]

    v_min = np.min(slice_2d)
    v_max = np.max(slice_2d)
    fig, ax = plt.subplots(ncols=num_thresholds + 1, figsize=(num_thresholds + 1, 11), dpi=300)

    ax[0].imshow(slice_2d, cmap='gray') # gray 0 -> черный, 1 -> белый
    ax[0].set_title(f"{axis} = {k}")
    ax[0].axis('off')

    for i, t in enumerate(np.linspace(v_min, v_max, num_thresholds)): # 20 порогов от 0 до 1
        ax[i+1].axis("off")
        ax[i+1].imshow(slice_2d <= t, cmap='gray_r', vmin=0, vmax=1)
        ax[i+1].set_title(f"{t:.2f}")
    plt.tight_layout()
    plt.show()


def visualize_cubical_complex_faces(img_filtered, num_thresholds = 10, title = 'cubical_complex', title_2 = "Только поры"):
      v_min = np.min(img_filtered)
      v_max = np.max(img_filtered)
      fig, ax = plt.subplots(ncols=num_thresholds + 1, figsize=(num_thresholds + 1, 11), dpi=300)

      visualize_grayscale_faces(img_filtered, f'cubical_complex_pore_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
      ax[0].imshow(mpimg.imread(f'/tmp/{title}.png'))
      ax[0].set_title(title_2)
      ax[0].axis('off')

      for i, t in enumerate(np.linspace(v_min, v_max, num_thresholds)):
          vis_values = np.where(img_filtered <= t, 0, 1)
          visualize_grayscale_faces(vis_values, f'{title}_{i+1}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
          ax[i+1].imshow(mpimg.imread(f'/tmp/{title}_{i+1}.png'))
          ax[i+1].set_title(f"{t:.2f}")
          ax[i+1].axis("off")
      plt.tight_layout()
      plt.show()

In [ ]:
show_cubical_complex_slice(grayscale_only_pore[0], 84, num_thresholds = 10)
show_cubical_complex_slice(grayscale_only_rock[0], 84, num_thresholds = 10)
show_cubical_complex_slice(grayscale_all[0], 84, num_thresholds = 20)





visualize_cubical_complex_faces(grayscale_only_pore[0], title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Только поры")
visualize_cubical_complex_faces(grayscale_only_rock[0], title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Только порода")
visualize_cubical_complex_faces(grayscale_all[0], 20, title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Всё")




# Векторизация персистентных диаграмм